In [0]:
from pyspark.sql.functions import *
dbutils.widgets.text("batch_id", "")
batch_id = dbutils.widgets.get("batch_id")
print(batch_id)

100


In [0]:
#read metadata for this batch
master_tbl_df=spark.sql(f"select * from metadata.master_tble where batch_id='{batch_id}'")
display(master_tbl_df)

batch_id,source_name,data_load_strategy,source_tbl_name,bronze_tbl_name,silver_tbl_name,watermark_column
100,azure_sql,FullLoad,dbo.product,bronze.product,silver.product,2000-09-07T15:23:08.535Z


In [0]:
#extract values once(avoid multiple collect calls)
row=master_tbl_df.first()
src_tbl_name_dynamic=row['source_tbl_name']
bronze_tbl_name_dynamic=row['bronze_tbl_name']
silver_tbl_name_dynamic=row['silver_tbl_name']
data_load_strategy_dynamic=row['data_load_strategy']
prev_modified_date_dynamic=row['watermark_column']
#read data from source table
print("Source:",src_tbl_name_dynamic)
print("Bronze:",bronze_tbl_name_dynamic)
print("Silver:",silver_tbl_name_dynamic)
print("Strategy:",data_load_strategy_dynamic)
print("prev Watermark:",prev_modified_date_dynamic)

Source: dbo.product
Bronze: bronze.product
Silver: silver.product
Strategy: FullLoad
prev Watermark: 2000-09-07 15:23:08.535000


In [0]:
#jdbc connection
server   = "dm-db.database.windows.net"
database = "Sales"
username = "dmuser"
password = "Dipti123@+"

src_url = f"jdbc:sqlserver://{server}:1433;database={database};user={username};password={password}"


In [0]:
if data_load_strategy_dynamic=="FullLoad":
    source_df=(
    spark.read.format("jdbc")
    .option("url",src_url)
    .option("dbtable",src_tbl_name_dynamic)
    .load()
    )
    
    #compute max modified
    src_max_modified_date=source_df.agg(max("ModifiedDate")).first()[0]
    print("New Watermark:",src_max_modified_date)
   
    source_df_add_col=source_df.withColumn("load_date",current_timestamp())

    source_df_add_col.write.mode("overwrite").saveAsTable(bronze_tbl_name_dynamic)
    source_df_add_col.write.mode("overwrite").saveAsTable(silver_tbl_name_dynamic)

    spark.sql(f"""
            UPDATE metadata.master_tble
            SET watermark_column='{src_max_modified_date}'
            WHERE batch_id='{batch_id}'
            """)
else:
    print("invalid batch_id or unsupported strategy")

New Watermark: 2025-11-13 00:00:00


In [0]:
%sql
SELECT * FROM bronze.product;

ProductID,Name,StandardCost,ModifiedDate,load_date
501,Laptop Pro 15,1200.00,2025-11-05T00:00:00.000Z,2025-11-30T03:58:14.716Z
502,Smartphone X,800.00,2025-11-06T00:00:00.000Z,2025-11-30T03:58:14.716Z
503,Wireless Headphones,150.00,2025-11-07T00:00:00.000Z,2025-11-30T03:58:14.716Z
504,null,500.00,null,2025-11-30T03:58:14.716Z
505,Tablet Z,-300.00,2025-11-08T00:00:00.000Z,2025-11-30T03:58:14.716Z
506,Laptop Pro 15,1200.00,2025-11-09T00:00:00.000Z,2025-11-30T03:58:14.716Z
507,Gaming Console Y,400.00,2025-11-10T00:00:00.000Z,2025-11-30T03:58:14.716Z
508,Smartwatch S,250.00,2025-11-11T00:00:00.000Z,2025-11-30T03:58:14.716Z
509,Monitor UltraWide,350.00,2025-11-12T00:00:00.000Z,2025-11-30T03:58:14.716Z
510,Keyboard Mechanical,90.00,2025-11-13T00:00:00.000Z,2025-11-30T03:58:14.716Z
